# 챗봇 구현

In [1]:
!pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu beautifulsoup4 transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [2]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace

/tmp/ipykernel_1022/3903406430.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [3]:
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs


문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="\n출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n\n\n[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

3

In [5]:
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question:
{question}

#Context:
{context}

#Answer:"""
)

In [7]:
llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct", # gemma-2-2b-it 접근 제한 모델이라 다른 모델 사용
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 512, "temperature": 0.1, "return_full_text": False},
)
llm = ChatHuggingFace(llm=llm_pipeline)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
# 체인을 생성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [9]:
answer = rag_chain.invoke("부영그룹의 출산 장려 정책에 대해 설명해주세요.")
print(answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


부영그룹의 출산 장려 정책은 다음과 같습니다:

1. **연령별 급여**: 2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고, 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.

2. **주택 제공**: 세 아이 모두를 갖춘 직원에게 국민주택을 제공합니다.

3. **세금 부담 감소**: 회사가 제공하는 출산장려금은 받는 직원들의 세금 부담을 고려해 정부가 면세해달라는 제안도 나왔습니다.

이 정책은 부영그룹의 경영 전략 중 하나로, 저출생 문제를 해결하기 위한 다양한 방법들을 시도하고 있습니다. 이러한 정책들은 부영그룹의 직원들에게 긍정적인 영향을 미치며, 사회적 분위기를 변화시키는 데 기여하고 있습니다.


In [10]:
answer = rag_chain.invoke("부영그룹은 출산 직원에게 얼마의 지원을 제공하나요?")
print(answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


부영그룹은 출산 직원에게 1억 원을 지원합니다.


In [11]:
answer = rag_chain.invoke("정부의 저출생 대책을 bullet points 형식으로 작성해 주세요.")
print(answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. **부모 급여**: 매달 100만원을 주는 부모 급여를 제공합니다.
2. **첫만남 이용권**: 0세 아이에게 100만원을 주며, 이를 통해 첫 만남을 즐길 수 있도록 지원합니다.
3. **아동수당**: 0세부터 18세까지 100만원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
4. **1년 동안 1520만원의 지원**: 0세부터 18세까지 1520만원을 지원하며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
5. **1억원의 지원**: 매달 1억원을 주는 파격적인 저출생 정책을 시행합니다.
6. **1억원의 지원**: 0세부터 18세까지 1억원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
7. **1억원의 지원**: 0세부터 18세까지 1억원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
8. **1억원의 지원**: 0세부터 18세까지 1억원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
9. **1억원의 지원**: 0세부터 18세까지 1억원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.
10. **1억원의 지원**: 0세부터 18세까지 1억원을 주며, 이를 통해 아이 돌릴 수 있는 돈을 마련합니다.


In [12]:
answer = rag_chain.invoke("부영그룹의 임직원 숫자는 몇명인가요?")
print(answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


부영그룹의 임직원 숫자는 문서에서 명확히 언급되지 않았습니다. 따라서 해당 정보를 찾을 수 없습니다.


# 챗봇 구현 - 다른 문서

In [13]:
!pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu beautifulsoup4 transformers accelerate

In [14]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from operator import itemgetter
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [15]:
loader = WebBaseLoader(
    web_paths=("https://www.ibm.com/kr-ko/think/topics/agentic-rag",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(["h2", "h3", "p"])
    ),
)

docs = loader.load()
print(docs[0].page_content[:1000])


        


  
  
    
    작성자

  





    


    Staff writerStaff Editor, AI ModelsIBM Think에이전틱 RAG는 AI 에이전트 를 사용하여 검색 증강 생성(RAG) 을 최적화하는 방식입니다. 에이전틱 RAG 시스템은 RAG 파이프라인에 AI 에이전트를 추가하여 적응성과 정확성을 높입니다. 기존 RAG 시스템과 비교할 때, 에이전틱 RAG는 대규모 언어 모델(LLM) 이 여러 소스에서 정보 검색 을 수행하고 더 복잡한 워크플로를 처리할 수 있습니다.RAG란 무엇인가요?검색 증강 생성은 생성형 AI 모델을 외부 지식 기반과 연결하는 인공 지능(AI) 애플리케이션입니다. 지식 기반의 데이터는 더 많은 컨텍스트로 사용자 쿼리를 보강하여 LLM이 더 정확한 응답을 생성할 수 있도록 합니다. RAG를 사용하면 LLM이 미세 조정 없이 도메인 특화 컨텍스트에서 더 정확하게 작동할 수 있습니다. RAG 지원 AI 모델은 학습 데이터에만 전적으로 의존하지 않고 API 및 기타 데이터 소스에 대한 연결을 통해 실시간으로 현재 데이터에 액세스할 수 있습니다. 표준 RAG 파이프라인은 다음과 같은 두 가지 AI 모델을 포함합니다. 정보 검색 구성 요소는 일반적으로 검색할 데이터가 포함된 벡터 데이터베이스와 쌍을 이루는 임베딩 모델입니다. 생성형 AI 구성 요소는 일반적으로 LLM입니다. 임베딩 모델은 자연어 사용자 쿼리에 대한 응답으로 쿼리를 벡터 임베딩으로 변환한 다음, 지식 기반에서 유사한 데이터를 검색합니다. AI 시스템은 검색된 데이터를 사용자 쿼리와 결합하여 컨텍스트 인식 응답을 생성합니다.에이전틱 AI란 무엇인가요?에이전틱 AI는 스스로 행동 방침을 결정하고 수행할 수 있는 AI의 한 유형입니다. 게시 시점에 사용할 수 있는 대부분의 에이전트는 함수 호출 능력이 있는 LLM으로, 도구를 호출하여 작업을 수행할 수 있습니다. 이론적으로 AI 에이전트는 다음과 같은 세 가지 중요한 특성을 가진 LLM

In [16]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

12

In [17]:
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)과 이전 대화(chat history)를 참고하여 질문(question)에 답하는 것입니다.
만약 주어진 문맥에서 답을 찾을 수 없다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요. 답변은 핵심만 3~5문장 이내로 간결하게 작성하세요.

#Previous Chat History:
{chat_history}

#Question:
{question}

#Context:
{context}

#Answer:"""
)

In [19]:
llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    task="text-generation",
    device=0,
    pipeline_kwargs={"max_new_tokens": 1024, "temperature": 0.1, "return_full_text": False},
)
llm = ChatHuggingFace(llm=llm_pipeline)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
store = {}  # 세션별 대화 기록 저장소

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
config = {"configurable": {"session_id": "test1"}}

print(rag_with_history.invoke({"question": "에이전틱 RAG가 기존 RAG보다 나은 점은 뭐야?"}, config=config))
print("-" * 50)
print(rag_with_history.invoke({"question": "그럼 반대로, 단점은 뭐야?"}, config=config))
print("-" * 50)
print(rag_with_history.invoke({"question": "지금까지 말한 장점과 단점을 각각 간단하게 정리해줘"}, config=config))

/tmp/ipykernel_1022/1614218717.py:5: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = ChatMessageHistory()
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


에이전틱 RAG은 유연성, 적응성, 확장성, 멀티모달 처리 능력 등 기존 RAG 시스템에 비해 여러 가지 중요한 개선 사항을 가지고 있습니다. 특히, 여러 외부 데이터 소스로부터 데이터를 가져올 수 있고, 다양한 상황에서 변화하는 요구사항에 맞춰 동작하며, 다양한 애플리케이션 분야에서 활용 가능합니다. 또한, 멀티에이전트 시스템을 통해 여러 AI 모델이 협업하여 결과를 최적화할 수 있습니다.
--------------------------------------------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


에이전틱 RAG의 단점으로는 데이터 소스의 제한, 요구사항에 대한 적응성 부족, 멀티에이전트 시스템의 복잡성 등이 있습니다. 특정 데이터 소스에서만 데이터를 가져올 수 있으므로 다양성에 한계가 있으며, 변화하는 요구사항에 대한 즉각적인 반응이 필요할 때는 부족할 수 있습니다. 또한, 여러 AI 모델이 협업해야 하는 멀티에이전트 시스템을 구축하는 것은 복잡하고 시간이 많이 걸릴 수 있습니다.
--------------------------------------------------
장점: 
- 유연성, 적응성, 확장성
- 여러 외부 데이터 소스로부터 데이터를 가져올 수 있음
- 다양한 애플리케이션 분야에서 활용 가능
- 멀티에이전트 시스템을 통해 여러 AI 모델이 협업하여 결과를 최적화할 수 있음

단점: 
- 특정 데이터 소스에서만 데이터를 가져올 수 있으므로 다양성에 한계가 있음
- 변화하는 요구사항에 대한 즉각적인 반응이 부족할 수 있음
- 멀티에이전트 시스템을 구축하는 것이 복잡하고 시간이 많이 걸림


In [23]:
config = {"configurable": {"session_id": "chat1"}}

while True:
    question = input("질문 (종료: q): ")
    if question.lower() == "q":
        break
    answer = rag_with_history.invoke({"question": question}, config=config)
    print(f"챗봇: {answer}\n")

질문 (종료: q): 에이전틱 rag가 뭐야


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


챗봇: 에이전틱 RAG은 기존 RAG 시스템과 비교하여 유연성, 적응성, 정확성, 확장성, 멀티모달 처리 능력을 갖추고 있습니다. 이를 통해 다양한 외부 데이터 소스를 활용하고, 변화하는 상황에 맞춰 응답을 제공하며, 정확한 결과를 얻기 위해 프롬프트 엔지니어링을 수행할 수 있습니다. 또한 여러 AI 모델이 협업하여 작업을 수행함으로써 멀티에이전트 시스템을 구축할 수 있습니다.

질문 (종료: q): 기존 rag랑 뭐가 달라


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


챗봇: 에이전틱 RAG은 기존 RAG 시스템보다 유연성, 적응성, 정확성, 확장성, 멀티모달 처리 능력을 갖추고 있습니다. 이를 통해 다양한 외부 데이터 소스를 활용하고, 변화하는 상황에 맞춰 응답을 제공하며, 정확한 결과를 얻기 위해 프롬프트 엔지니어링을 수행할 수 있습니다. 또한 여러 AI 모델이 협업하여 작업을 수행함으로써 멀티에이전트 시스템을 구축할 수 있습니다.

질문 (종료: q): 방금 말한 차이 중 정확성에 대해 더 자세히 설명해줘


...

챗봇: 쿼리 계획 에이전트는 복잡한 작업을 계획하고 실행하며, 이전 작업의 데이터를 참조하여 향후 워크플로에 정보를 제공합니다. 이는 AI 오케스트레이션에서 각 AI 에이전트가 다른 작업에 적절히 배치되고 조정되는 방식을 의미합니다. 따라서 쿼리 계획 에이전트는 에이전틱 RAG 시스템 내에서 AI 오케스트레이션을 돕습니다.

질문 (종료: q): 지금까지 대화 내용을 3줄로 요약해줘


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


챗봇: 에이전틱 RAG은 다양한 외부 데이터 소스를 활용하고, 변화하는 상황에 맞춰 응답을 제공하며, 정확한 결과를 얻기 위해 프롬프트 엔지니어링을 수행할 수 있습니다. 하지만 모든 상황에서 에이전틱 RAG이 항상 가장 좋다는 것은 아닙니다. 쿼리 계획 에이전트는 복잡한 작업을 계획하고 실행하며, 이전 작업의 데이터를 참조하여 향후 워크플로에 정보를 제공합니다. 이는 AI 오케스트레이션에서 각 AI 에이전트가 다른 작업에 적절히 배치되고 조정되는 방식을 의미합니다.

질문 (종료: q): 에이전틱 rag를 처음 만든 회사는 어디야?


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


챗봇: 에이전틱 RAG을 처음 만든 회사는 공개적으로 알려진 정보는 아닙니다. 제공된 정보에서는 해당 회사의 명칭이 언급되지 않았습니다. 에이전틱 RAG은 기존 RAG 시스템과 비교하여 유연성, 적응성, 정확성, 확장성, 멀티모달 처리 능력을 갖추고 있습니다.

질문 (종료: q): q
